# MemGPT-Style Tiered Memory: An Agent That Manages Its Own Context


## Definition

**MemGPT** (the paper behind what later became the [Letta](https://www.letta.com/) framework) reframes an LLM agent's
context window as the "RAM" of a virtual memory system, and gives the agent an "OS"-like set of
**self-editing memory tools** so it can manage what lives in that limited RAM versus what gets
paged out to unbounded "disk" storage. The key idea is not a smarter retriever bolted on from the
outside — it is that **the agent itself decides**, mid-conversation, what is important enough to
keep close (core memory), what can be archived, and when to go fetch something back.

This notebook builds a deliberately simplified version of that idea from scratch (no external
framework), inspired by the `31_memgpt` example in
[FareedKhan-dev/all-agentic-architectures](https://github.com/FareedKhan-dev/all-agentic-architectures).

## High-Level Workflow

1. **Main context ("RAM")** — a small, fixed-size window made of:
   - a **core memory** block (a handful of persistent key facts, always in the prompt), and
   - the **last N conversation messages** (a sliding window).
2. **External context ("disk")** — an unbounded **archival memory** store. When the sliding window
   overflows, the oldest messages are evicted here automatically (like paging out of RAM).
3. **Self-editing tools** bound to the LLM via `bind_tools`:
   - `core_memory_append(block, content)` — write a new durable fact.
   - `core_memory_replace(block, old, new)` — correct/update a durable fact.
   - `archival_memory_insert(content)` — push something out to external storage.
   - `archival_memory_search(query)` — pull something back into context for this turn.
4. On every turn the agent sees core memory + the current window, and can call any of the above
   tools before producing its final answer.

## When to Use

- Long-running assistants / companions where a user has durable facts (preferences, ongoing
  projects) that must survive far longer than any fixed context window.
- Agents that need to hold a very long conversation or document trail without paying full
  attention-cost for every past token on every turn.
- Situations where you want the *agent* — not a hand-written heuristic — deciding what is worth
  remembering permanently.

## Strengths / Weaknesses

**Strengths**
- Decouples "what the agent can reason over right now" from "what the agent knows in total" —
  effectively unbounded long-term memory with a small, cheap working set.
- Memory management is transparent and inspectable: you can print core memory and archival
  storage at any point and see exactly what the agent chose to keep.
- The agent can correct itself (`core_memory_replace`) instead of accumulating stale facts.

**Weaknesses**
- Relies on the LLM reliably *choosing* to call memory tools at the right moments — it can forget
  to archive an important fact or fail to search when it should.
- Naive archival search (here: simple keyword overlap) can miss relevant memories; production
  systems back this with a real vector store.
- Extra tool-call round trips add latency and cost versus a single flat prompt.


In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

from helpers import get_llm


### What We Are Going to Do

We will implement two small in-memory classes that stand in for MemGPT's "RAM" and "disk":

- `MainContext` — holds the core memory blocks (a dict of named facts) and a bounded list of
  recent messages. When a new message pushes the list past `max_messages`, the oldest message is
  evicted and handed to archival storage.
- `ArchivalMemory` — an unbounded list acting as our stand-in for a vector DB / file store, with a
  naive keyword-overlap `search` (a real system would embed and use nearest-neighbor search).

Then we define four `@tool`-decorated functions the LLM can call directly, bind them to the model
via `llm.bind_tools([...])`, and write a small agent loop that:

1. Builds a system prompt from current core memory + a description of the memory tools.
2. Sends core memory + the sliding window of messages to the LLM.
3. Executes any tool calls the LLM makes (looping until it stops calling tools).
4. Appends the final answer to the window (triggering eviction if needed).

Finally we run a multi-turn conversation long enough to see facts written to core memory, older
turns evicted to archival storage, and a later turn forcing an `archival_memory_search` call to
answer correctly.


In [ ]:
# ============ TIERED MEMORY: MAIN CONTEXT ("RAM") + ARCHIVAL MEMORY ("DISK") ============


class ArchivalMemory:
    """Unbounded external store standing in for a vector DB / file store.

    Retrieval here is a naive keyword-overlap score purely for demonstration purposes; a real
    MemGPT-style system would embed each entry and use nearest-neighbor search instead.
    """

    def __init__(self):
        self.store: list[dict] = []

    def insert(self, content: str) -> None:
        self.store.append({"id": len(self.store), "content": content})

    def search(self, query: str, top_k: int = 3) -> list[str]:
        query_terms = set(query.lower().split())
        scored = []
        for item in self.store:
            content_terms = set(item["content"].lower().split())
            overlap = len(query_terms & content_terms)
            if overlap > 0:
                scored.append((overlap, item["content"]))
        scored.sort(key=lambda pair: -pair[0])
        return [content for _, content in scored[:top_k]]

    def __len__(self) -> int:
        return len(self.store)


class MainContext:
    """Fixed-size working context: a small core-memory dict + a sliding window of messages.

    When the window exceeds `max_messages`, the oldest message is evicted to archival memory --
    mirroring MemGPT's "paging out of RAM" behavior.
    """

    def __init__(self, max_messages: int = 6):
        self.max_messages = max_messages
        self.core_memory: dict[str, str] = {
            "persona": "I am a helpful, concise travel-planning assistant.",
            "human": "",
        }
        self.messages: list[dict] = []

    def add_message(self, role: str, content: str, archival: ArchivalMemory) -> None:
        self.messages.append({"role": role, "content": content})
        while len(self.messages) > self.max_messages:
            evicted = self.messages.pop(0)
            archival.insert(f"[{evicted['role']}] {evicted['content']}")

    def core_memory_str(self) -> str:
        return "\n".join(f"- {block}: {value}" for block, value in self.core_memory.items() if value)


In [ ]:
main_context = MainContext(max_messages=6)
archival_memory = ArchivalMemory()


In [ ]:
# ============ SELF-EDITING MEMORY TOOLS BOUND TO THE LLM ============


@tool
def core_memory_append(block: str, content: str) -> str:
    """Append content to a core memory block (e.g. 'human' or 'persona').

    Use this to permanently remember an important fact about the user or the task -- something
    that should always stay visible in context, no matter how long the conversation gets.
    """
    current = main_context.core_memory.get(block, "")
    main_context.core_memory[block] = (current + " " + content).strip()
    return f"Core memory block '{block}' updated."


@tool
def core_memory_replace(block: str, old_content: str, new_content: str) -> str:
    """Replace old_content with new_content inside a core memory block.

    Use this to correct or update a previously stored fact rather than letting stale
    information accumulate.
    """
    current = main_context.core_memory.get(block, "")
    if old_content not in current:
        return f"'{old_content}' not found in core memory block '{block}'."
    main_context.core_memory[block] = current.replace(old_content, new_content)
    return f"Core memory block '{block}' updated."


@tool
def archival_memory_insert(content: str) -> str:
    """Insert content into the unbounded archival memory store (the external 'disk').

    Use this to explicitly save information that does not need to stay in the main context
    window but should still be recoverable later via archival_memory_search.
    """
    archival_memory.insert(content)
    return "Inserted into archival memory."


@tool
def archival_memory_search(query: str) -> str:
    """Search archival memory for content relevant to the query.

    Use this whenever you need information that is not visible in your current core memory or
    main context window -- e.g. something from earlier in the conversation that has since been
    evicted.
    """
    results = archival_memory.search(query)
    if not results:
        return "No relevant results found in archival memory."
    return "\n".join(f"- {r}" for r in results)


memory_tools = [core_memory_append, core_memory_replace, archival_memory_insert, archival_memory_search]
tool_map = {t.name: t for t in memory_tools}


In [ ]:
llm = get_llm()
llm_with_tools = llm.bind_tools(memory_tools)


### The Agent Loop

Each turn we rebuild the system prompt from *current* core memory (so edits made a moment ago are
immediately visible), send it together with the sliding message window, and then loop over any
tool calls the model makes -- feeding results back in as `ToolMessage`s -- until the model responds
with plain text. Only then do we record the exchange into `MainContext` (which may trigger an
eviction to archival memory).


In [ ]:
# ============ AGENT LOOP: PROMPT ASSEMBLY, TOOL-CALL HANDLING, EVICTION ============


def build_system_prompt() -> str:
    return f"""You are a MemGPT-style agent with tiered memory.

Core Memory (always visible, persists across the whole conversation):
{main_context.core_memory_str() or "(empty)"}

Your main context window only holds your last {main_context.max_messages} messages. Older
messages are automatically evicted to archival memory, so if a user refers to something you
can no longer see, call archival_memory_search to look it up.

You have these memory-management tools available -- use them proactively:
- core_memory_append(block, content): permanently remember an important fact about the user/task.
- core_memory_replace(block, old_content, new_content): correct a previously stored fact.
- archival_memory_insert(content): explicitly push information out to external storage.
- archival_memory_search(query): retrieve something from external storage back into context.
"""


def _to_lc_message(msg: dict):
    return HumanMessage(content=msg["content"]) if msg["role"] == "user" else AIMessage(content=msg["content"])


def run_turn(user_input: str, verbose: bool = True) -> str:
    main_context.add_message("user", user_input, archival_memory)

    messages = [SystemMessage(content=build_system_prompt())]
    messages += [_to_lc_message(m) for m in main_context.messages]

    response = llm_with_tools.invoke(messages)

    while getattr(response, "tool_calls", None):
        messages.append(response)
        for call in response.tool_calls:
            fn = tool_map[call["name"]]
            result = fn.invoke(call["args"])
            if verbose:
                print(f"  [tool call] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        response = llm_with_tools.invoke(messages)

    main_context.add_message("assistant", response.content, archival_memory)
    return response.content


def print_state(label: str) -> None:
    print(f"\n=== {label} ===")
    print("Core memory:")
    print(main_context.core_memory_str() or "  (empty)")
    print(f"Main context window ({len(main_context.messages)}/{main_context.max_messages} messages):")
    for m in main_context.messages:
        print(f"  [{m['role']}] {m['content'][:80]}")
    print(f"Archival memory ({len(archival_memory)} entries):")
    for item in archival_memory.store:
        print(f"  #{item['id']}: {item['content'][:80]}")


### What We Are Going to Do

We now drive a multi-turn conversation designed to exercise all three tier-transition behaviors:

1. **Turn 1** — the user states a durable preference (a strict dietary restriction) that the
   agent should proactively write to `core_memory_append`.
2. **Turn 2** — the user mentions a one-off fact (their trip budget) that is *not* important
   enough for core memory, but will matter later. It only lives in the main context window for
   now.
3. **Turns 3-5** — filler small talk that pushes the window past `max_messages=6`, evicting the
   turn 1 and turn 2 messages out to archival memory automatically.
4. **Turn 6** — the user asks about the budget mentioned in turn 2. It is no longer in the window,
   so the agent must call `archival_memory_search` to recover it before it can answer correctly.

We print core memory + archival state after key turns so the tier transitions are visible.


In [ ]:
print("USER: Hi, I'm planning a trip to Kyoto in November. Please remember that my dietary "
      "restriction is strictly vegetarian.")
answer = run_turn(
    "Hi, I'm planning a trip to Kyoto in November. Please remember that my dietary restriction "
    "is strictly vegetarian."
)
print("ASSISTANT:", answer)
print_state("After turn 1 (durable fact should be in core memory)")


In [ ]:
print("USER: My budget for the trip is $2500.")
answer = run_turn("My budget for the trip is $2500.")
print("ASSISTANT:", answer)
print_state("After turn 2 (budget is only in the main context window so far)")


In [ ]:
filler_turns = [
    "What's the weather usually like in Kyoto in November?",
    "Any flight booking tips for international travel?",
    "What neighborhoods are good to stay in for first-time visitors?",
]

for turn_text in filler_turns:
    print("USER:", turn_text)
    answer = run_turn(turn_text)
    print("ASSISTANT:", answer)

print_state("After filler turns 3-5 (turn 1 & 2 messages should now be evicted to archival)")


In [ ]:
print("USER: By the way, what was the budget I mentioned earlier for my Kyoto trip?")
answer = run_turn("By the way, what was the budget I mentioned earlier for my Kyoto trip?")
print("ASSISTANT:", answer)
print_state("After turn 6 (answer should only be correct if archival_memory_search was called)")


### Discussion of the Output

- After **turn 1**, `core_memory["human"]` should contain the vegetarian dietary restriction --
  the agent chose to write it because it is durable, identity-level information about the user.
- After **turn 2**, the $2500 budget exists only as a plain message in the main context window --
  it is not important enough on its own to warrant a core memory write, so it is exactly the kind
  of fact that will be silently evicted once the window fills up.
- After the **filler turns**, `print_state` should show the window holding only the three most
  recent exchanges, with the turn 1 and turn 2 messages now materialized as entries in
  `archival_memory.store` -- memory has physically moved tiers.
- On **turn 6**, the budget question can no longer be answered from the visible window or core
  memory, so a well-behaved model calls `archival_memory_search("budget Kyoto trip")`, gets the
  evicted message back, and only then answers "$2500". If the tool call log for this turn is
  empty, that is a visible failure mode of relying on the LLM to decide *when* to search --
  exactly the weakness called out above.


## Summary

- We built a simplified MemGPT-style memory system with two explicit tiers: a small, always-
  visible **core memory** + sliding message window ("RAM"), and an unbounded **archival memory**
  store ("disk") that automatically absorbs evicted messages.
- The defining MemGPT idea is that **the agent manages its own memory** through real, callable
  tools (`core_memory_append`, `core_memory_replace`, `archival_memory_insert`,
  `archival_memory_search`) bound via `llm.bind_tools(...)` -- not a hand-written heuristic
  deciding what to keep.
- Printing core memory and archival storage at each step made the tier transitions concrete: a
  durable fact promoted to core memory, a transient fact evicted to archival storage, and a later
  turn forced to explicitly retrieve it back into context.
- This is one of several memory-architecture notebooks under `Agentic_Memory_Architectures/` --
  siblings covering **Graph Memory**, a **Voyager-style skill library**, and **Agent Workflow
  Memory** are being authored separately as companion notebooks in this same folder.
